In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

def scrape_action_movies(target=1461):
    options = webdriver.ChromeOptions()
    options.add_argument('--window-size=1920,1080')
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    # URL filtered for 2024 Action Movies
    url = "https://www.imdb.com/search/title/?title_type=feature&release_date=2024-01-01,2024-12-31&genres=action&adult=include"
    driver.get(url)
    
    movie_list = []
    
    # Simple loop to load more until target is reached
    while len(movie_list) < target:
        items = driver.find_elements(By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item")
        for item in items[len(movie_list):]:
            try:
                title = item.find_element(By.CSS_SELECTOR, "h3.ipc-title__text").text.split('. ', 1)[-1]
                story = item.find_element(By.CSS_SELECTOR, ".ipc-html-content-inner-div").text
                movie_list.append({"Title": title, "Storyline": story})
                if len(movie_list) >= target: break
            except: continue
        
        # Click "Load More"
        try:
            btn = driver.find_element(By.CSS_SELECTOR, "button.ipc-see-more__button")
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(3)
        except: break

    pd.DataFrame(movie_list).to_csv("action_movies.csv", index=False)
    driver.quit()

if __name__ == "__main__":
    scrape_action_movies()